In [1]:
print('test')

test


In [17]:
import pandas as pd

stops = pd.read_csv("./data_inputs/gtfs_m/stops.txt")  


# Result: DataFrame with stop_id, stop_name, stop_lat, stop_lon
print(stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]].head())

   stop_id           stop_name   stop_lat   stop_lon
0   101014  WILLIS AV/E 138 ST  40.808930 -73.922855
1   101015  WILLIS AV/E 140 ST  40.810634 -73.921632
2   101017  WILLIS AV/E 144 ST  40.813111 -73.919823
3   101018  WILLIS AV/E 146 ST  40.814404 -73.918954
4   101066       3 AV/E 150 ST  40.816142 -73.917547


In [18]:
import json, os

os.makedirs("./outputs", exist_ok=True)

geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [row.stop_lon, row.stop_lat]},
            "properties": {"stop_id": row.stop_id, "stop_name": row.stop_name},
        }
        for row in stops.itertuples()
    ],
}

with open("./outputs/stops.geojson", "w") as f:
    json.dump(geojson, f)

print(f"Saved {len(geojson['features'])} stops to outputs/stops.geojson")

Saved 1828 stops to outputs/stops.geojson


In [19]:
routes = pd.read_csv("./data_inputs/gtfs_m/routes.txt")
shapes = pd.read_csv("./data_inputs/gtfs_m/shapes.txt")
trips = pd.read_csv("./data_inputs/gtfs_m/trips.txt")[["route_id", "shape_id"]].drop_duplicates()

route_shapes = trips.merge(routes[["route_id", "route_short_name"]], on="route_id")

features = []
for shape_id, group in shapes.sort_values("shape_pt_sequence").groupby("shape_id"):
    coords = list(zip(group.shape_pt_lon, group.shape_pt_lat))
    match = route_shapes.loc[route_shapes.shape_id == shape_id]
    route_id = match.iloc[0].route_id if not match.empty else None
    route_name = match.iloc[0].route_short_name if not match.empty else None
    features.append({
        "type": "Feature",
        "geometry": {"type": "LineString", "coordinates": coords},
        "properties": {"shape_id": shape_id, "route_id": route_id, "route_name": route_name},
    })

routes_geojson = {"type": "FeatureCollection", "features": features}

os.makedirs("./outputs", exist_ok=True)
with open("./outputs/routes.geojson", "w") as f:
    json.dump(routes_geojson, f)

print(f"Saved {len(features)} route shapes to outputs/routes.geojson")

Saved 160 route shapes to outputs/routes.geojson


In [20]:
import math
from shapely.geometry import Point, LineString
from shapely.strtree import STRtree
from shapely.ops import nearest_points

# Build shapely geometries for all route shapes
route_lines = [
    (feat["properties"]["route_id"], feat["properties"]["shape_id"], LineString(feat["geometry"]["coordinates"]))
    for feat in routes_geojson["features"]
]

# Spatial index for fast nearest-line lookup
tree = STRtree([line for _, _, line in route_lines])

def bearing(lat1, lon1, lat2, lon2):
    """Bearing in degrees (0=N, 90=E) from point 1 → point 2."""
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    x = math.sin(lon2 - lon1) * math.cos(lat2)
    y = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(lon2 - lon1)
    return (math.degrees(math.atan2(x, y)) + 360) % 360

snapped_rows = []
for row in stops.itertuples():
    stop_pt = Point(row.stop_lon, row.stop_lat)
    idx = tree.nearest(stop_pt)
    route_id, shape_id, line = route_lines[idx]
    snapped_pt = nearest_points(line, stop_pt)[0]
    hdg = bearing(snapped_pt.y, snapped_pt.x, row.stop_lat, row.stop_lon)
    snapped_rows.append({
        "stop_id": row.stop_id,
        "stop_name": row.stop_name,
        "stop_lat": row.stop_lat,
        "stop_lon": row.stop_lon,
        "snapped_lat": snapped_pt.y,
        "snapped_lon": snapped_pt.x,
        "heading": round(hdg, 1),
        "route_id": route_id,
        "shape_id": shape_id,
    })

snapped_df = pd.DataFrame(snapped_rows)

# Save as GeoJSON (geometry = snapped point)
snapped_geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [r["snapped_lon"], r["snapped_lat"]]},
            "properties": {k: v for k, v in r.items() if k not in ("snapped_lat", "snapped_lon")},
        }
        for r in snapped_rows
    ],
}

with open("./outputs/snapped_stops.geojson", "w") as f:
    json.dump(snapped_geojson, f)

print(f"Saved {len(snapped_rows)} snapped stops to outputs/snapped_stops.geojson")
snapped_df[["stop_id", "stop_name", "snapped_lat", "snapped_lon", "heading", "route_id"]].head()

Saved 1828 snapped stops to outputs/snapped_stops.geojson


,stop_id,stop_name,snapped_lat,snapped_lon,heading,route_id
0,101014,WILLIS AV/E 138 ST,40.808992,-73.922945,132.4,M125
1,101015,WILLIS AV/E 140 ST,40.810696,-73.921723,131.9,M125
2,101017,WILLIS AV/E 144 ST,40.813170,-73.919906,133.1,M125
3,101018,WILLIS AV/E 146 ST,40.814428,-73.918990,130.7,M125
4,101066,3 AV/E 150 ST,40.816111,-73.917529,336.0,M125


In [39]:
import requests
from dotenv import load_dotenv

load_dotenv(".env.local")
api_key = os.environ["API_KEY"]

os.makedirs("./outputs/street_view", exist_ok=True)

def fetch_street_view(snapped_lat, snapped_lon, heading, stop_id, size="640x640", fov=90, pitch=0):
    params = {
        "size": size,
        "location": f"{snapped_lat},{snapped_lon}",
        "heading": heading,
        "fov": fov,
        "pitch": pitch,
        "key": api_key,
    }
    # r = requests.get("https://maps.googleapis.com/maps/api/streetview", params=params)
    # r.raise_for_status()
    # path = f"./outputs/street_view/{stop_id}.jpg"
    # with open(path, "wb") as f:
    #     f.write(r.content)
    # return path

for row in snapped_df.sample(12).itertuples():
    fetch_street_view(row.snapped_lat, row.snapped_lon, row.heading, row.stop_id)
    sv_link = f"https://www.google.com/maps/@?api=1&map_action=pano&viewpoint={row.snapped_lat},{row.snapped_lon}&heading={row.heading}&pitch=0&fov=90"
    print(f" {sv_link}\n")

print("Done.")

 https://www.google.com/maps/@?api=1&map_action=pano&viewpoint=40.76652207837385,-73.98672398804962&heading=314.0&pitch=0&fov=90

 https://www.google.com/maps/@?api=1&map_action=pano&viewpoint=40.7894813844667,-73.93974058251281&heading=198.0&pitch=0&fov=90

 https://www.google.com/maps/@?api=1&map_action=pano&viewpoint=40.713438089106795,-73.97825273964058&heading=193.0&pitch=0&fov=90

 https://www.google.com/maps/@?api=1&map_action=pano&viewpoint=40.78353382209087,-73.95053257494783&heading=133.9&pitch=0&fov=90

 https://www.google.com/maps/@?api=1&map_action=pano&viewpoint=40.73245784053902,-73.98485382313307&heading=314.1&pitch=0&fov=90

 https://www.google.com/maps/@?api=1&map_action=pano&viewpoint=40.82695270134228,-73.9389036409396&heading=314.4&pitch=0&fov=90

 https://www.google.com/maps/@?api=1&map_action=pano&viewpoint=40.80629663332637,-73.96500795401032&heading=133.4&pitch=0&fov=90

 https://www.google.com/maps/@?api=1&map_action=pano&viewpoint=40.822822013782826,-73.93787

In [43]:
sv_url = lambda row: f"https://www.google.com/maps/@?api=1&map_action=pano&viewpoint={row.snapped_lat},{row.snapped_lon}&heading={row.heading}&pitch=0&fov=90"

export_df = snapped_df[["stop_id", "stop_name", "stop_lat", "stop_lon", "snapped_lat", "snapped_lon", "heading", "route_id"]].copy()
export_df["streetview_url"] = [sv_url(row) for row in snapped_df.itertuples()]

export_df.to_csv("./outputs/snapped_stops.csv", index=False)
print(f"Saved {len(export_df)} rows to outputs/snapped_stops.csv")
export_df.head()

Saved 1828 rows to outputs/snapped_stops.csv


,stop_id,stop_name,stop_lat,stop_lon,snapped_lat,snapped_lon,heading,route_id,streetview_url
0,101014,WILLIS AV/E 138 ST,40.808930,-73.922855,40.808992,-73.922945,132.4,M125,https://www.google.com/maps/@?api=1&map_action...
1,101015,WILLIS AV/E 140 ST,40.810634,-73.921632,40.810696,-73.921723,131.9,M125,https://www.google.com/maps/@?api=1&map_action...
2,101017,WILLIS AV/E 144 ST,40.813111,-73.919823,40.813170,-73.919906,133.1,M125,https://www.google.com/maps/@?api=1&map_action...
3,101018,WILLIS AV/E 146 ST,40.814404,-73.918954,40.814428,-73.918990,130.7,M125,https://www.google.com/maps/@?api=1&map_action...
4,101066,3 AV/E 150 ST,40.816142,-73.917547,40.816111,-73.917529,336.0,M125,https://www.google.com/maps/@?api=1&map_action...


In [42]:
stops.shape

(1828, 9)